# MMDS — automated cloud pipeline
Choose **Runtime → Run all** on a free CPU Colab runtime. No code edits are required. The default workflow installs missing or mismatched compatible dependencies, runs tests, acquires the authorized Elliptic release, validates it, computes PySpark graph features, trains chronological baselines and saves models, metrics, processed tables, logs and a Markdown report to private Google Drive.

Use a Google account, connect a hosted runtime, review the publisher's dataset terms, and authorize Drive when prompted. These are the unavoidable manual actions. Nothing executes on your Mac or iPhone. [Full guide](https://github.com/ojassgambira/MMDS/blob/fix/colab-runtime-pip/docs/colab.md).

Python 3.11–3.13 and Java 17/21 are supported; Colab uses its existing Python and pip without ensurepip or venv creation. Required package versions may be updated in that runtime; models run in fresh subprocesses so cached notebook imports are not reused. Other cloud environments retain isolated venvs. A future unsupported runtime stops with an explanation. The notebook is committed without outputs: **Colab execution and real-data results are not claimed**.

The default source is the documented public PyG mirror of `ellipticco/elliptic-data-set` (CC BY-NC-ND 4.0 per publisher documentation); no Kaggle secret is needed. No retries, alternate unapproved mirrors, GPUs or paid resources are used. Default processing covers the bounded original release (203,769 transactions), with two CPU workers and three small baseline models. The whole worker has a one-hour budget.

Predictors use only graph information available after each source step completes. Train: steps 1–34; validation: 35–39; test: 40–49. Unknown labels are excluded from fitting/metrics. Source anonymous aggregates are excluded. This is retrospective end-of-step classification; it does not establish live label availability.

Optional form controls below choose temporary storage or a private cloud configuration JSON. Defaults require no changes. Keep the tab active on iPhone. Drive authorization can pause Run all until approved. A free runtime may disconnect; rerunning restores completed checkpoints with matching code/configuration/input hashes.

In [ ]:
#@title Run MMDS automatically
import os
from pathlib import Path
import subprocess
import sys
from google.colab import drive
from IPython.display import Markdown, display

storage_mode = "Google Drive" #@param ["Google Drive", "Temporary Colab"]
config_path = "" #@param {type:"string"}

workspace = Path('/content/mmds-automated')
repo = Path('/content/MMDS-automated')
failure_details = None
pipeline_succeeded = False
pipeline_started = False
try:
    if not Path('/content').is_dir():
        raise RuntimeError('Run this notebook only in a hosted Google Colab runtime.')
    if not (3, 11) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError('Supported Python versions are 3.11–3.13; select a compatible hosted runtime.')
    os.environ['MMDS_COLAB_RUNTIME'] = '1'
    url = 'https://github.com/ojassgambira/MMDS.git'
    if not repo.exists():
        subprocess.run(['git', 'clone', url, str(repo)], check=True, capture_output=True, text=True)
    origin = subprocess.check_output(['git', '-C', str(repo), 'remote', 'get-url', 'origin'], text=True)
    if origin.strip() != url:
        raise RuntimeError('Unexpected repository origin')
    if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Cloud checkout has edits; preserve them and use a clean checkout')
    branch = 'fix/colab-runtime-pip'
    refs = subprocess.run(['git', 'ls-remote', '--heads', url, branch], check=True,
                          capture_output=True, text=True).stdout
    ref = branch if refs.strip() else 'main'
    for operation in (['fetch', 'origin', ref], ['checkout', '--detach', 'FETCH_HEAD']):
        subprocess.run(['git', '-C', str(repo), *operation], check=True, capture_output=True, text=True)
    sys.path.insert(0, str(repo / 'src'))
    from mmd_fraud.diagnostics import failure_details
    command = [sys.executable, str(repo / 'scripts/run_cloud.py'), '--workspace', str(workspace)]
    if storage_mode == 'Google Drive':
        drive.mount('/content/drive')
        command += ['--storage', '/content/drive/MyDrive/MMDS/automated']
    if config_path.strip():
        command += ['--config', str(Path(config_path).expanduser())]
    pipeline_started = True
    subprocess.run(command, check=True, stderr=subprocess.PIPE, text=True)
    report_path = Path((workspace / 'latest-report.txt').read_text())
    display(Markdown(report_path.read_text()))
    pipeline_succeeded = True
    print('Temporary run artifacts:', report_path.parent)
    if storage_mode == 'Google Drive':
        print('Persistent artifacts: My Drive/MMDS/automated/runs/' + report_path.parent.name)
    else:
        print('Temporary artifacts disappear when the runtime is reclaimed.')
except Exception as error:
    if failure_details is not None:
        print(failure_details(workspace, error, prefer_recorded=pipeline_started))
    else:
        # Repository/Drive setup can fail before the shared diagnostics are importable.
        print('MMDS setup failed:', type(error).__name__, str(error))
        for field in ('stdout', 'stderr'):
            output = getattr(error, field, None)
            if output:
                print(str(output)[-8000:])
        print('Available log tails may be from an earlier attempt; setup failed before worker launch.')
        for name in ('bootstrap.log', 'runtime.log', 'worker.log'):
            logfile = workspace / name
            try:
                with logfile.open('rb') as handle:
                    handle.seek(0, 2)
                    handle.seek(max(0, handle.tell() - 8000))
                    print(name, handle.read(8000).decode(errors='replace'))
            except OSError:
                print(name, '(log unavailable; setup stopped before this log was created)')
    print('Pipeline status: FAILED. No successful dataset access or model execution is claimed.')


## Completion and recovery
The report states success only after validation, model fitting and checkpoint persistence finish. It includes precision, recall, F1, PR-AUC (average precision), confusion matrices, split counts and stage timings. A failed run does not imply successful acquisition or model evaluation. If authentication or source access fails, read the report; do not repeatedly run a denied download.

For an existing authorized Kaggle copy, put the three publisher CSVs in a private Drive folder and supply a JSON configuration through the form (see the guide). Kaggle authentication, if required by its official client, belongs in Colab Secrets, never code or Git. Default mirror acquisition needs no account credential. Do not share raw/derived data, model pickle files or tokens publicly. Load only models you trust.

Completed stages resume from Drive when code, Python/dependencies, settings and input hashes match. Interrupted stages restart; free Colab cannot guarantee uninterrupted execution. Dataset bytes and models remain outside the repository.
Failures display their stage, underlying cause and bounded tails from bootstrap, runtime, worker and check logs. Missing logs are identified explicitly. Authorization/access denials show a single next-action instruction; denied downloads are never retried.
